<a href="https://colab.research.google.com/github/AllyApitchaya/msc-adr-prediction/blob/main/notebooks/03_baseline_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/MSc_ADR_Project'
print(f"Project path: {PROJECT_PATH}")

Mounted at /content/drive
Project path: /content/drive/MyDrive/MSc_ADR_Project


In [2]:
import pandas as pd
import os

# Disproportionality methods (PRR, ROR, BCPNN) require raw report
# counts, not the unique drug-ADR pairs used for the ML models.
# We therefore load the deduplicated FAERS dataset produced by
# notebook 01.
faers_path = os.path.join(PROJECT_PATH, 'data', 'processed',
                          'faers_2020_2022_diabetes_adr.csv')
faers = pd.read_csv(faers_path)

print(f"Loaded {len(faers):,} FAERS reports")
print(f"Columns: {list(faers.columns)}\n")
print(f"Unique drugs:    {faers['drugname'].nunique()}")
print(f"Quarters:        {sorted(faers['quarter'].unique())}")

Loaded 80,958 FAERS reports
Columns: ['primaryid', 'caseid', 'drugname', 'role_cod', 'pt', 'quarter']

Unique drugs:    482
Quarters:        ['2020q1', '2020q2', '2020q3', '2020q4', '2021q1', '2021q2', '2021q3', '2021q4', '2022q1', '2022q2', '2022q3', '2022q4']


In [3]:
# Use the mapped dataset, which has drug names normalised to the
# 15 antidiabetic generic drugs. This file still contains one row
# per report (not unique pairs), so it is suitable for counting.
mapped_path = os.path.join(PROJECT_PATH, 'data', 'processed',
                           'drug_adr_with_structures.csv')
faers = pd.read_csv(mapped_path)

print(f"Loaded {len(faers):,} reports")
print(f"Columns: {list(faers.columns)}\n")
print(f"Unique drugs: {faers['drug'].nunique()}")
print(f"Drug names:   {sorted(faers['drug'].unique())}\n")

# Restrict to the training period, so the disproportionality
# baselines use the same data window as the ML models. This keeps
# the comparison fair: every model sees only training-period data.
TRAIN_QUARTERS = ['2020q1', '2020q2', '2020q3',
                  '2020q4', '2021q1', '2021q2']

faers_train = faers[faers['quarter'].isin(TRAIN_QUARTERS)].copy()

print(f"Reports in training period: {len(faers_train):,}")
print(f"Training quarters: {TRAIN_QUARTERS}")

Loaded 80,958 reports
Columns: ['drug', 'pubchem_cid', 'smiles', 'adr', 'caseid', 'quarter']

Unique drugs: 15
Drug names:   ['acarbose', 'canagliflozin', 'dapagliflozin', 'empagliflozin', 'glimepiride', 'glipizide', 'glyburide', 'linagliptin', 'metformin', 'nateglinide', 'pioglitazone', 'repaglinide', 'rosiglitazone', 'saxagliptin', 'sitagliptin']

Reports in training period: 40,020
Training quarters: ['2020q1', '2020q2', '2020q3', '2020q4', '2021q1', '2021q2']


In [4]:
# Build a 2x2 contingency table for every drug-ADR pair observed
# in the training period. These four counts (a, b, c, d) are the
# basis for all three disproportionality measures.
#
#                  ADR present     ADR absent
#   drug present        a               b
#   drug absent         c               d

# Total reports in the training window
N = len(faers_train)

# a: reports for each specific (drug, adr) pair
pair_counts = (faers_train
               .groupby(['drug', 'adr'])
               .size()
               .reset_index(name='a'))

# Row totals: all reports mentioning each drug  (a + b)
drug_totals = (faers_train
               .groupby('drug')
               .size()
               .rename('drug_total'))

# Column totals: all reports mentioning each ADR  (a + c)
adr_totals = (faers_train
              .groupby('adr')
              .size()
              .rename('adr_total'))

# Attach totals to each pair
ct = pair_counts.merge(drug_totals, on='drug')
ct = ct.merge(adr_totals, on='adr')

# Derive the remaining cells of the 2x2 table
ct['b'] = ct['drug_total'] - ct['a']
ct['c'] = ct['adr_total'] - ct['a']
ct['d'] = N - ct['a'] - ct['b'] - ct['c']

print(f"Total training reports (N): {N:,}")
print(f"Drug-ADR pairs in contingency table: {len(ct):,}\n")

# Show the table for a known pair as a sanity check
example = ct[(ct['drug'] == 'metformin') &
             (ct['adr'] == 'Lactic acidosis')]
print("Example - metformin / Lactic acidosis:")
print(example[['drug', 'adr', 'a', 'b', 'c', 'd']].to_string(index=False))

Total training reports (N): 40,020
Drug-ADR pairs in contingency table: 4,193

Example - metformin / Lactic acidosis:
     drug             adr    a     b  c    d
metformin Lactic acidosis 2509 33184 38 4289


In [5]:
import numpy as np

# Proportional Reporting Ratio (PRR)
#   PRR = [a / (a + b)] / [c / (c + d)]
#
# A continuity correction of 0.5 is added to every cell. This
# avoids division by zero when a drug-ADR pair is rare, and is a
# standard adjustment in disproportionality analysis.
CORRECTION = 0.5

a = ct['a'] + CORRECTION
b = ct['b'] + CORRECTION
c = ct['c'] + CORRECTION
d = ct['d'] + CORRECTION

ct['prr'] = (a / (a + b)) / (c / (c + d))

# A pair is flagged as a signal under the common PRR >= 2 rule
ct['prr_signal'] = ct['prr'] >= 2

print("PRR computed for all drug-ADR pairs.\n")
print(f"Pairs flagged as signals (PRR >= 2): "
      f"{ct['prr_signal'].sum():,} of {len(ct):,}\n")

# Sanity check on the known pair
example = ct[(ct['drug'] == 'metformin') &
             (ct['adr'] == 'Lactic acidosis')]
print("Example - metformin / Lactic acidosis:")
print(example[['drug', 'adr', 'a', 'c', 'prr']].to_string(index=False))

print("\nTop 10 pairs by PRR:")
top = ct.nlargest(10, 'prr')[['drug', 'adr', 'a', 'prr']]
print(top.to_string(index=False))

PRR computed for all drug-ADR pairs.

Pairs flagged as signals (PRR >= 2): 1,798 of 4,193

Example - metformin / Lactic acidosis:
     drug             adr    a  c      prr
metformin Lactic acidosis 2509 38 7.903483

Top 10 pairs by PRR:
         drug                                      adr  a          prr
rosiglitazone                                 Kyphosis  1 15005.250000
rosiglitazone                           Limb deformity  1 15005.250000
rosiglitazone                       Multiple fractures  1 15005.250000
    glyburide         Counterfeit product administered 17 13179.811321
rosiglitazone                               Osteopenia  1  5001.750000
rosiglitazone              Spinal compression fracture  1  5001.750000
     acarbose Drug effective for unapproved indication  1  2262.396226
     acarbose                 Pneumatosis intestinalis  1  2262.396226
     acarbose                          Zinc deficiency  1  2262.396226
rosiglitazone                   Bone density decreas

In [6]:
# Refine the PRR signal definition. Following Evans et al. (2001),
# a signal requires both PRR >= 2 and a minimum of 3 reports for
# the pair (a >= 3). The a >= 3 threshold filters out spurious
# high-PRR pairs that are driven by a single chance report.
MIN_REPORTS = 3

ct['prr_signal'] = (ct['prr'] >= 2) & (ct['a'] >= MIN_REPORTS)

n_signal = ct['prr_signal'].sum()
print(f"PRR signals (PRR >= 2 and a >= {MIN_REPORTS}): "
      f"{n_signal:,} of {len(ct):,}")
print(f"(Previously, without the a >= {MIN_REPORTS} rule: 1,798)\n")

# Top signals now that low-evidence pairs are excluded
print("Top 10 PRR signals (a >= 3):")
top = (ct[ct['prr_signal']]
       .nlargest(10, 'prr')[['drug', 'adr', 'a', 'prr']])
print(top.to_string(index=False))

PRR signals (PRR >= 2 and a >= 3): 533 of 4,193
(Previously, without the a >= 3 rule: 1,798)

Top 10 PRR signals (a >= 3):
         drug                              adr  a          prr
    glyburide Counterfeit product administered 17 13179.811321
 pioglitazone      Blood electrolytes abnormal  6   917.744186
 pioglitazone            Femoral neck fracture  6   917.744186
  glimepiride         Cerebral artery embolism  5   808.817505
  glimepiride        Hyporesponsive to stimuli  5   808.817505
  glimepiride                  Neuroglycopenia  5   808.817505
    glyburide            Product contamination 17   693.674280
  glimepiride                 Prerenal failure  4   661.759777
empagliflozin                     Genital rash  7   649.817276
 pioglitazone          Adult failure to thrive  4   635.361360


In [7]:
# Reporting Odds Ratio (ROR)
#   ROR = (a * d) / (b * c)
#
# ROR uses odds rather than proportions. The same 0.5 continuity
# correction is applied. As with PRR, a signal requires ROR >= 2
# and at least 3 reports for the pair.

# Corrected cells (reuse the 0.5 correction)
a = ct['a'] + CORRECTION
b = ct['b'] + CORRECTION
c = ct['c'] + CORRECTION
d = ct['d'] + CORRECTION

ct['ror'] = (a * d) / (b * c)

ct['ror_signal'] = (ct['ror'] >= 2) & (ct['a'] >= MIN_REPORTS)

print(f"ROR signals (ROR >= 2 and a >= {MIN_REPORTS}): "
      f"{ct['ror_signal'].sum():,} of {len(ct):,}\n")

# Sanity check on the known pair
example = ct[(ct['drug'] == 'metformin') &
             (ct['adr'] == 'Lactic acidosis')]
print("Example - metformin / Lactic acidosis:")
print(example[['drug', 'adr', 'a', 'prr', 'ror']].to_string(index=False))

print("\nTop 10 ROR signals (a >= 3):")
top = (ct[ct['ror_signal']]
       .nlargest(10, 'ror')[['drug', 'adr', 'a', 'ror']])
print(top.to_string(index=False))

ROR signals (ROR >= 2 and a >= 3): 533 of 4,193

Example - metformin / Lactic acidosis:
     drug             adr    a      prr      ror
metformin Lactic acidosis 2509 7.903483 8.425542

Top 10 ROR signals (a >= 3):
         drug                              adr  a          ror
    glyburide Counterfeit product administered 17 15785.790960
 pioglitazone      Blood electrolytes abnormal  6   928.529412
 pioglitazone            Femoral neck fracture  6   928.529412
    glyburide            Product contamination 17   830.643770
  glimepiride         Cerebral artery embolism  5   817.176858
  glimepiride        Hyporesponsive to stimuli  5   817.176858
  glimepiride                  Neuroglycopenia  5   817.176858
  glimepiride                 Prerenal failure  4   667.343662
empagliflozin                     Genital rash  7   655.251256
 pioglitazone          Adult failure to thrive  4   640.509468


In [8]:
# Bayesian Confidence Propagation Neural Network (BCPNN)
#
# BCPNN produces the Information Component (IC):
#   IC = log2( observed co-occurrence / expected co-occurrence )
#
# The expected co-occurrence assumes the drug and ADR are
# independent. A Bayesian-style 0.5 smoothing is applied, which
# shrinks the IC towards 0 when evidence is weak. This makes
# BCPNN more conservative than PRR/ROR for rare pairs.

# Observed and marginal counts
a_obs = ct['a']
drug_total = ct['a'] + ct['b']   # reports for the drug
adr_total = ct['a'] + ct['c']    # reports for the ADR

# Expected count if drug and ADR were independent
expected = (drug_total * adr_total) / N

# IC with 0.5 smoothing on both observed and expected terms
ct['ic'] = np.log2((a_obs + 0.5) / (expected + 0.5))

# A pair is flagged as a signal when IC > 0 and a >= 3
ct['ic_signal'] = (ct['ic'] > 0) & (ct['a'] >= MIN_REPORTS)

print(f"BCPNN signals (IC > 0 and a >= {MIN_REPORTS}): "
      f"{ct['ic_signal'].sum():,} of {len(ct):,}\n")

# Sanity check on the known pair
example = ct[(ct['drug'] == 'metformin') &
             (ct['adr'] == 'Lactic acidosis')]
print("Example - metformin / Lactic acidosis:")
print(example[['drug', 'adr', 'a', 'prr', 'ror', 'ic']]
      .to_string(index=False))

print("\nTop 10 BCPNN signals by IC (a >= 3):")
top = (ct[ct['ic_signal']]
       .nlargest(10, 'ic')[['drug', 'adr', 'a', 'ic']])
print(top.to_string(index=False))

BCPNN signals (IC > 0 and a >= 3): 1,214 of 4,193

Example - metformin / Lactic acidosis:
     drug             adr    a      prr      ror       ic
metformin Lactic acidosis 2509 7.903483 8.425542 0.143363

Top 10 BCPNN signals by IC (a >= 3):
         drug                               adr   a       ic
    glyburide  Counterfeit product administered  17 5.006007
    glyburide             Product contamination  17 4.944772
dapagliflozin Euglycaemic diabetic ketoacidosis 103 4.518476
 pioglitazone                    Bladder cancer  21 4.484978
  repaglinide                Hypoglycaemic coma  15 4.307557
dapagliflozin             Diabetic ketoacidosis  95 4.228502
empagliflozin                       Pollakiuria  36 4.150565
dapagliflozin                      Ketoacidosis  33 4.031521
 pioglitazone                            Oedema  15 3.892381
empagliflozin         Urine ketone body present  10 3.571573


In [9]:
# Build feature tables for the ML baseline. Each drug-ADR pair in
# the train/val/test splits is described by the disproportionality
# statistics computed from the training period.
#
# Pairs not seen in the training period have no statistics; these
# are filled with neutral defaults (ratios = 1, IC = 0, counts = 0),
# representing "no evidence of disproportionality".

FEATURE_COLS = ['prr', 'ror', 'ic', 'a', 'drug_total', 'adr_total']
NEUTRAL = {'prr': 1.0, 'ror': 1.0, 'ic': 0.0,
           'a': 0, 'drug_total': 0, 'adr_total': 0}

processed_dir = os.path.join(PROJECT_PATH, 'data', 'processed')

# Statistics lookup, keyed by (drug, adr)
stats = ct[['drug', 'adr'] + FEATURE_COLS]


def build_features(split_name):
    """Attach training-period disproportionality features to a
    train/val/test split.
    """
    df = pd.read_csv(os.path.join(processed_dir, f'{split_name}.csv'))
    merged = df.merge(stats, on=['drug', 'adr'], how='left')
    # Fill pairs unseen in training with neutral values
    merged = merged.fillna(NEUTRAL)
    return merged


train_feat = build_features('train')
val_feat = build_features('val')
test_feat = build_features('test')

print("Feature tables built:\n")
for name, df in [('train', train_feat),
                 ('val', val_feat),
                 ('test', test_feat)]:
    unseen = (df['a'] == 0).sum()
    print(f"  {name:6s} {len(df):,} pairs  "
          f"({unseen:,} unseen in training period)")

print(f"\nFeature columns: {FEATURE_COLS}")
print("\nSample of train features:")
print(train_feat[['drug', 'adr', 'prr', 'ic', 'label']]
      .head(5).to_string(index=False))

Feature tables built:

  train  8,240 pairs  (4,120 unseen in training period)
  val    4,912 pairs  (3,190 unseen in training period)
  test   9,016 pairs  (6,812 unseen in training period)

Feature columns: ['prr', 'ror', 'ic', 'a', 'drug_total', 'adr_total']

Sample of train features:
       drug                          adr      prr        ic  label
  glipizide                 Constipation 2.400036  0.695650      1
nateglinide Hyperplastic cholecystopathy 1.000000  0.000000      0
  metformin        Hepatitis cholestatic 0.121253 -0.606447      1
  glyburide          Portal hypertension 1.000000  0.000000      0
  glyburide                  Hypotension 1.898652  0.226725      1


In [10]:
# Colab does not always preinstall xgboost; install it for this
# session. (Packages are lost when the runtime disconnects.)
!pip install xgboost -q
print("xgboost installed.")

xgboost installed.


In [11]:
import xgboost as xgb
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             accuracy_score, f1_score)

# Prepare feature matrix X and label vector y for each split.
def to_xy(df):
    X = df[FEATURE_COLS].astype(float)
    y = df['label'].astype(int)
    return X, y

X_train, y_train = to_xy(train_feat)
X_val,   y_val   = to_xy(val_feat)
X_test,  y_test  = to_xy(test_feat)

# XGBoost classifier. The validation set is used for early
# stopping so the model does not overfit the training data.
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    early_stopping_rounds=20,
    random_state=42,
)

model.fit(X_train, y_train,
          eval_set=[(X_val, y_val)],
          verbose=False)

# Predict on the held-out test set
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

# Evaluate with several metrics
results = {
    'AUROC':    roc_auc_score(y_test, y_prob),
    'AUPRC':    average_precision_score(y_test, y_prob),
    'Accuracy': accuracy_score(y_test, y_pred),
    'F1':       f1_score(y_test, y_pred),
}

print("XGBoost baseline - test set performance:\n")
for metric, value in results.items():
    print(f"  {metric:9s}: {value:.4f}")

# Feature importance shows which signals XGBoost relied on
print("\nFeature importance:")
importance = sorted(zip(FEATURE_COLS, model.feature_importances_),
                    key=lambda x: x[1], reverse=True)
for feat, imp in importance:
    print(f"  {feat:12s}: {imp:.4f}")

XGBoost baseline - test set performance:

  AUROC    : 0.7008
  AUPRC    : 0.6828
  Accuracy : 0.7008
  F1       : 0.5980

Feature importance:
  drug_total  : 0.5074
  a           : 0.2714
  adr_total   : 0.2212
  prr         : 0.0000
  ror         : 0.0000
  ic          : 0.0000


In [12]:
# Summarise the baseline phase and save results for later
# comparison against the GNN/BioBERT model.

results_dir = os.path.join(PROJECT_PATH, 'results')
os.makedirs(results_dir, exist_ok=True)

# 1. Save the disproportionality table (contingency + PRR/ROR/IC)
disp_cols = ['drug', 'adr', 'a', 'b', 'c', 'd',
             'prr', 'prr_signal', 'ror', 'ror_signal',
             'ic', 'ic_signal']
disp_path = os.path.join(results_dir, 'disproportionality_results.csv')
ct[disp_cols].to_csv(disp_path, index=False)
print(f"Saved disproportionality table -> {disp_path}")
print(f"  {len(ct):,} drug-ADR pairs\n")

# 2. Save the XGBoost test-set metrics
xgb_results = pd.DataFrame([results])
xgb_path = os.path.join(results_dir, 'baseline_xgboost_metrics.csv')
xgb_results.to_csv(xgb_path, index=False)
print(f"Saved XGBoost metrics -> {xgb_path}\n")

# 3. Summary of signal counts across disproportionality methods
print("=" * 50)
print("BASELINE SUMMARY")
print("=" * 50)
print("\nDisproportionality signal counts "
      f"(of {len(ct):,} pairs):")
print(f"  PRR   (PRR >= 2, a >= 3): {ct['prr_signal'].sum():,}")
print(f"  ROR   (ROR >= 2, a >= 3): {ct['ror_signal'].sum():,}")
print(f"  BCPNN (IC  > 0,  a >= 3): {ct['ic_signal'].sum():,}")

print("\nXGBoost baseline (test set):")
for metric, value in results.items():
    print(f"  {metric:9s}: {value:.4f}")

print("\nThis is the performance bar for the GNN/BioBERT model.")
print("The main model must exceed AUROC = "
      f"{results['AUROC']:.4f} to demonstrate added value.")

Saved disproportionality table -> /content/drive/MyDrive/MSc_ADR_Project/results/disproportionality_results.csv
  4,193 drug-ADR pairs

Saved XGBoost metrics -> /content/drive/MyDrive/MSc_ADR_Project/results/baseline_xgboost_metrics.csv

BASELINE SUMMARY

Disproportionality signal counts (of 4,193 pairs):
  PRR   (PRR >= 2, a >= 3): 533
  ROR   (ROR >= 2, a >= 3): 533
  BCPNN (IC  > 0,  a >= 3): 1,214

XGBoost baseline (test set):
  AUROC    : 0.7008
  AUPRC    : 0.6828
  Accuracy : 0.7008
  F1       : 0.5980

This is the performance bar for the GNN/BioBERT model.
The main model must exceed AUROC = 0.7008 to demonstrate added value.
